<img src="https://images.squarespace-cdn.com/content/v1/67351b6c456f5a62d1c1bca2/17a93d03-0cae-49bf-b152-8dfd0cdb9d7d/Quantum+Rings+Logo+Main+White.png?format=300w" alt="Quantum Rings" width="150" align="right"/>

# Quantum Fourier Transform
The Quantum Fourier Transform (QFT) is a key quantum algorithm that performs a discrete Fourier transform on the amplitudes of a quantum state. It is a crucial component in various quantum algorithms, notably Shor's algorithm for factoring large numbers, as it allows quantum computers to analyze periodicities in data more efficiently than classical computers. Unlike the classical Fourier Transform, the QFT operates on a superposition of states, exploiting quantum parallelism to perform its computations exponentially faster. The QFT essentially transforms quantum states into a new basis, where each state represents a different frequency component of the original state. This transformation enables quantum algorithms to leverage interference patterns, making QFT a foundational tool in quantum computing.

This sample code implements the Quantum Fourier Transform, following the explanation in the Quantum Rings documentation here:
https://portal.quantumrings.com/doc/QFT.html

## Setup

Replace the credential placeholders in the provider cell below with your Quantum Rings API token and account email address, available from the [Quantum Rings portal](https://portal.quantumrings.com).

In [ ]:
%%capture
%pip install QuantumRingsLib matplotlib numpy

In [ ]:
import QuantumRingsLib
from QuantumRingsLib import QuantumRegister, AncillaRegister, ClassicalRegister, QuantumCircuit
from QuantumRingsLib import QuantumRingsProvider
from QuantumRingsLib import job_monitor
from QuantumRingsLib import JobStatus
from matplotlib import pyplot as plt
import numpy as np
import math

In [ ]:
provider = QuantumRingsProvider(token="YOUR_TOKEN_HERE", name="YOUR_EMAIL_ADDRESS")
backend = provider.get_backend("scarlet_quantum_rings")
shots = 100

provider.active_account()

In [ ]:
def set_reg(qc, input, q, n):
    """
    Sets a quantum register with an input value.

    Args:
        qc (QuantumCircuit):
            The quantum circuit to use

        input (int):
            The number to be stored in the qubit register

        q (QuantumRegister):
            The qubit register which is to be programmed with the input number

        n (int):
            The width of the qubit register to use.

    Returns:
        None

    """

    for i in range (0, n):
        if ((1 << i) & input ):
            qc.x(q[i])
    return



def add_cct(qc,a, b, n):
    """
    The adder circuit in the Fourier Basis

    Args:
        qc (QuantumCircuit):
            The quantum circuit

        a (QuantumRegister):
            The source register

        b (QuantumRegister):
            The target register

        n (int):
            The number of qubits in the registers to use

    Returns:
        None

    """

    while (n):
        for i in range(n, 0, -1):
            qc.cu1(math.pi/2**(n - i), a[i-1], b[n-1])
        n -= 1
        qc.barrier()
    return



def iqft_cct(qc, b, n):
    """
    The inverse QFT circuit

    Args:

        qc (QuantumCircuit):
                The quantum circuit

        b (QuantumRegister):
                The target register

        n (int):
                The number of qubits in the registers to use

    Returns:
        None

    """

    for i in range (n):
        for j in range (1, i+1):
            # for inverse transform, we have to use negative angles
            qc.cu1(  -math.pi / 2** ( i -j + 1 ), b[j - 1], b[i])
        # the H transform should be done after the rotations
        qc.h(b[i])
    qc.barrier()
    return


def qft_cct(qc, b, n):
    """
    The Forward QFT circuit

    Args:

        qc (QuantumCircuit):
                The quantum circuit

        b (QuantumRegister):
                The target register

        n (int):
                The number of qubits in the registers to use

    Returns:
        None

    """

    while (n):
        qc.h(b[n-1])
        for i in range (n-1, 0, -1):
            qc.cu1(math.pi / 2** (n - i ), b[i - 1], b[n-1])
        n -= 1
        qc.barrier()
    return


def add_qft(qc, input1, input2, a, b, n):
    """
    The adder using QFT.

    Args:

        qc (QuantumCircuit):
            The quantum circuit

        input1 (int):
            The first number

        input2 (int):
            The second number to add.

       a (QuantumRegister):
            The source register

        b (QuantumRegister):
            The target register. Computed value is stored in this register

        n (int):
            The number of qubits in the registers to use

    Returns:
        None

    """
    set_reg(qc, input1, a, n)
    set_reg(qc, input2, b, n)

    qft_cct(qc, b, n)
    add_cct(qc, a, b, n)
    iqft_cct(qc, b, n)
    return

def plot_histogram (counts, title=""):
    """
    Plots the histogram of the counts

    Args:

        counts (dict):
            The dictionary containing the counts of states

        titles (str):
            A title for the graph.

    Returns:
        None

    """
    fig, ax = plt.subplots(figsize =(10, 7))
    plt.xlabel("States")
    plt.ylabel("Counts")
    mylist = [key for key, val in counts.items() for _ in range(val)]

    unique, inverse = np.unique(mylist, return_inverse=True)
    bin_counts = np.bincount(inverse)

    plt.bar(unique, bin_counts)

    maxFreq = max(counts.values())
    plt.ylim(ymax=np.ceil(maxFreq / 10) * 10 if maxFreq % 10 else maxFreq + 10)
    # Show plot
    plt.title(title)
    plt.show()
    return

Enter values you'd like, but if you enter 111111 and 1. The result 1000000 is expected.

In [ ]:
# obtain the hidden string from the user
input_a = int(input("Enter the first number as the binary value: "), 2)
input_b = int(input("Enter the second number as the binary value: "), 2)

# determine the number of qubits required to represent the hidden string
# add one more qubit to handle the carry. This is essentially needed in the second bank, though.

numberofqubits = max(input_a.bit_length(), input_b.bit_length()) + 1

print("Number of qubits required in each register: ", numberofqubits)


q1 = QuantumRegister(numberofqubits , 'a')
q2 = QuantumRegister(numberofqubits , 'b')
c = ClassicalRegister(numberofqubits , 'c')
qc = QuantumCircuit(q1, q2, c)

add_qft(qc, input_a, input_b, q1, q2, numberofqubits)

for i in range (q2.size()):
    qc.measure(q2[i], i)

job = backend.run(qc, shots=shots)
job_monitor(job)
result = job.result()
counts = result.get_counts()

# clean-up the circuit components
del q1
del q2
del c
del qc

plot_histogram(counts,"")

# clean-up
del result
del job

In [ ]:
def c_add_cct(qc, a, b, offset, control, n):
    """
        The controlled adder circuit module.

        Args:

            qc (QuantumCircuit):
                The quantum circuit

            a (QuantumRegister):
                The source register

            b (QuantumRegister):
                The target register. Computed value is stored in this register

            offset (int):
                The starting qubit in register b to work from

            n (int):
                The number of qubits in the registers to use

        Returns:
            None

    """
    while (n):
        for i in range(n, 0, -1):
            ccu1(qc, math.pi/2**(n - i), control, a[i-1], b[n-1+offset])
        qc.barrier()
        n -= 1
    return



def c_iqft_cct (qc, b, offset, control, n):
    """
        The controlled Inverse-QFT.

        Args:

            qc (QuantumCircuit):
                The quantum circuit

            b (QuantumRegister):
                The target register.

            offset (int):
                The starting qubit in register b to work from

            control (int):
                The index number of the control qubit

            n (int):
                The number of qubits in the registers to use

        Returns:
            None

    """
    for i in range (n):
        for j in range (1, i+1):
            # for inverse transform, we have to use negative angles
            ccu1( qc, -math.pi / 2** ( i -j + 1 ), control, b[j - 1+offset], b[i+offset])
        # the H transform should be done after the rotations
        qc.ch(control, b[i+offset])
    qc.barrier()

    return



def c_qft_cct(qc, b, offset, control, n):
    """
        The controlled QFT.

        Args:

            qc (QuantumCircuit):
                The quantum circuit

            b (QuantumRegister):
                The target register.

            offset (int):
                The starting qubit in register b to work from

            control (int):
                The index number of the control qubit

            n (int):
                The number of qubits in the registers to use

        Returns:
            None

    """
    while (n):
        qc.ch(control, b[n + offset - 1])
        for i in range (n-1, 0, -1):
            ccu1(qc, math.pi / 2** (n - i), control, b[i - 1 + offset], b[n + offset - 1])
        n -= 1
    qc.barrier()

    return



def c_add_qft(qc, a, b, offset, control, n):
    """
        The controlled Adder using QFT.

        Args:

            qc (QuantumCircuit):
                The quantum circuit

            a (QuantumRegister):
                The source register

            b (QuantumRegister):
                The target register.

            offset (int):
                The starting qubit in register b to work from

            control (int):
                The index number of the control qubit

            n (int):
                The number of qubits in the registers to use

        Returns:
            None

    """
    c_qft_cct(qc, b, offset, control, n)
    c_add_cct(qc, a, b, offset, control, n)
    c_iqft_cct(qc, b, offset, control, n)
    return


def ccu1(qc, theta, q0, q1, q2):
    """
    The controlled Adder using QFT.

    Args:

        qc (QuantumCircuit):
            The quantum circuit

        theta (float):
            The rotational angle. See :ref:`QuantumCircuit.u1`

        q0 (int):
            The first control qubit.

        q1 (int):
            The second control qubit.

        q2 (int):
            The target qubit.


    Returns:
    None

    """
    qc.cu1(theta/2, q1, q2)
    qc.cx(q0, q1)
    qc.cu1(-theta/2, q1, q2)
    qc.cx(q0, q1)
    qc.cu1(theta/2, q0, q2)
    return

In [ ]:
def mult_cct(qc, input1, input2, a, b, s, n):
    """
    The quantum multiplier circuit using QFT
    This circuit calculates s = a * b.

    Args:

        qc (QuantumCircuit):
            The quantum circuit

        input1 (int):
            The multiplicand

        input2 (int):
            The multiplier

        s (QuantumRegister):
            The qubit register holding product a * b
            Note: s has two times the length of a and b

        a (QuantumRegister):
            The qubit register for multiplicand

        b (QuantumRegister):
            The qubit register for multiplier

        n (int)
            The length of the registers

    Returns:
        None.

    """
    set_reg(qc, input1, a, n)
    set_reg(qc, input2, b, n)

    for i in range (0, n):
        c_add_qft(qc, b, s, i,  a[i], n)

    return

In [ ]:
def test_mult( input1, input2):

    numberofqubits = max(input1.bit_length(), input2.bit_length(), 4)


    a = QuantumRegister(numberofqubits , 'a')
    b = QuantumRegister(numberofqubits , 'b')
    s = QuantumRegister(numberofqubits*2 , 's')
    c = ClassicalRegister(numberofqubits*2 , 'c')
    qc = QuantumCircuit(a, b, s, c)


    mult_cct(qc, input1, input2, a, b, s, numberofqubits)

    for i in range (numberofqubits*2):
        qc.measure(s[i],c[i])

    job = backend.run(qc, shots=shots)
    job_monitor(job) #, quiet = False)
    result = job.result()
    counts = result.get_counts()
    print (counts)
    if (1 < len(counts)):
        print("Error: More than one result!")
        myproduct = 0
    else:
        scounts = str(counts)
        myproduct = int("0b"+scounts[scounts.index("{")+2:scounts.index(":")-1], 2)

    # clean-up
    del qc
    del c
    del s
    del b
    del a
    del job
    del result

    return myproduct


shots = 10
input_a = int(input("Enter the multiplcand: "))
input_b = int(input("Enter the multiplier: "))

res = test_mult(input_a, input_b )
print ("Result is: ", res)